# Training Faster R-CNN with k-fold cross validation

The code in here is roughly based on this tutorial
https://pytorch.org/tutorials/intermediate/torchvision_tutorial.html


## Hyperparameters


In [ ]:
BATCH_SIZE = 4

# Wether to use the COCO pretrained Faster R-CNN
PRETRAINED = True

# Wether to use the AGAR dataset for pretraining
AGAR_TRAINING = False
EPOCHS_AGAR = 1

EPOCHS_OWN = 10
KFOLDS_OWN = 5

# Max norm for gradient clipping
MAX_NORM = 1

## Trainer class


In [ ]:
from pathlib import Path
from typing import Optional

import torch
from engine import evaluate, train_one_epoch
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter


def average_losses(loss_dicts: list[dict[str, torch.Tensor]]):
    loss_cls = torch.stack([loss["loss_classifier"] for loss in loss_dicts]).mean()
    loss_box_reg = torch.stack([loss["loss_box_reg"] for loss in loss_dicts]).mean()
    loss_objectness = torch.stack(
        [loss["loss_objectness"] for loss in loss_dicts]
    ).mean()
    loss_rpn_box_reg = torch.stack(
        [loss["loss_rpn_box_reg"] for loss in loss_dicts]
    ).mean()

    summed_loss = loss_cls + loss_box_reg + loss_objectness + loss_rpn_box_reg

    return {
        "classifier": loss_cls,
        "box_reg": loss_box_reg,
        "objectness": loss_objectness,
        "rpn_box_reg": loss_rpn_box_reg,
        "total": summed_loss,
    }


class Trainer:
    def __init__(
        self,
        model: torch.nn.Module,
        train_data: DataLoader,
        val_data: Optional[DataLoader],
        optimizer: torch.optim.Optimizer,
        gpu_id: int,
        writer: SummaryWriter,
        model_save_path: str,
    ) -> None:
        self.gpu_id = gpu_id
        self.model = model.to(gpu_id)
        self.train_data = train_data
        self.val_data = val_data
        self.optimizer = optimizer
        self.writer = writer

        self.should_val = val_data is not None

        self.model_save_path = model_save_path
        Path(model_save_path).parent.mkdir(parents=True, exist_ok=True)

    def _run_epoch(self, epoch, scaler):
        # train for one epoch, printing every 10 iterations
        train_losses = train_one_epoch(
            self.model,
            self.optimizer,
            self.train_data,
            self.gpu_id,
            epoch,
            scaler=scaler,
            print_freq=10,
            max_norm=MAX_NORM,
        )

        avg_train_loss_dict = average_losses(train_losses)
        self.writer.add_scalars("train_loss", avg_train_loss_dict, epoch)

        # check loss on validation set
        if self.should_val:
            with torch.no_grad():
                val_losses = train_one_epoch(
                    self.model,
                    self.optimizer,
                    self.val_data,
                    self.gpu_id,
                    epoch,
                    print_freq=10,
                    optimize=False,
                )

                avg_val_loss_dict = average_losses(val_losses)
                self.writer.add_scalars("val_loss", avg_val_loss_dict, epoch)

                # evaluate on the validation dataset
                coco_evaluator, mae, smape = evaluate(
                    self.model, self.val_data, device=self.gpu_id
                )

                stats = coco_evaluator.coco_eval["bbox"].stats

                avg_precision_iou_50 = stats[0]
                avg_recall_iou_50 = stats[8]

                self.writer.add_scalars(
                    "val_metrics",
                    {
                        "avg_precision_iou_50": avg_precision_iou_50,
                        "avg_recall_iou_50": avg_recall_iou_50,
                        "mae": mae,
                        "smape": smape,
                    },
                    epoch,
                )

        self.writer.flush()

    def _save_checkpoint(self, epoch):
        ckp = self.model.state_dict()
        torch.save(ckp, self.model_save_path)
        print(f"Epoch {epoch} | Training checkpoint saved at {self.model_save_path}")

    def train(self, max_epochs: int):
        scaler = torch.cuda.amp.GradScaler()

        for epoch in range(max_epochs):
            self._run_epoch(epoch, scaler)

            if epoch % 10 == 0:
                self._save_checkpoint(epoch)

        self._save_checkpoint(max_epochs)

## Data helpers


In [ ]:
import json
import utils
from torch.utils.data import Dataset
from dataset_agar import AgarBBDataset


# The box data for these IDs of the AGAR dataset have negative x values for the bounding boxes.
faulty_ids = ["5794", "11988"]


def load_id_list(path: str):
    with open(path) as f:
        list = json.load(f)
        list = [int(id) for id in list if id.isdigit() and id not in faulty_ids]

    return list


def prepare_agar_datasets():
    faulty_ids = ["5794", "11988"]

    train_list = load_id_list(
        "../AGAR_dataset/training_lists/higher_resolution_train.txt"
    )
    val_list = load_id_list("../AGAR_dataset/training_lists/higher_resolution_val.txt")

    print(f"Training on {len(train_list)} images.")
    print(f"Validating on {len(val_list)} images.")

    train_dataset = AgarBBDataset("../AGAR_dataset/dataset", train_list, is_train=True)
    val_dataset = AgarBBDataset("../AGAR_dataset/dataset", val_list, is_train=False)

    return train_dataset, val_dataset


def prepare_dataloader(dataset: Dataset, batch_size: int, is_train: bool):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=4,
        pin_memory=True,
        shuffle=is_train,
        collate_fn=utils.collate_fn,
    )

## Main function


In [ ]:
from datetime import datetime

from dataset_photobox import PhotoboxBBDataset
from model import get_model_instance_segmentation
from sklearn.model_selection import KFold
from torch.optim import AdamW


def main(device):
    start_time_str = datetime.now().strftime("%Y-%m-%d_%H-%M")

    # Agar Pretraining
    if AGAR_TRAINING:
        print("Training on AGAR dataset\n" + "-" * 10 + "\n")

        train_dataset, val_dataset = prepare_agar_datasets()

        train_loader = prepare_dataloader(train_dataset, BATCH_SIZE, is_train=True)
        val_dataset = prepare_dataloader(val_dataset, BATCH_SIZE, is_train=False)

        model = get_model_instance_segmentation(
            AgarBBDataset.num_classes,
            pretrained=PRETRAINED,
        )

        params = [p for p in model.parameters() if p.requires_grad]
        optimizer = AdamW(params, lr=0.005, weight_decay=0.01)

        writer = SummaryWriter(f"runs/{start_time_str}/agar")
        agar_model_path = f"artifacts/{start_time_str}/agar.pth"

        trainer = Trainer(
            model,
            train_loader,
            val_dataset,
            optimizer,
            device,
            writer,
            agar_model_path,
        )

        trainer.train(EPOCHS_AGAR)

    # K-fold cross validation
    kfold = KFold(n_splits=KFOLDS_OWN, shuffle=True)

    full_train_dataset = PhotoboxBBDataset(
        "../train_data", is_train=True, patch_augmentation=False
    )

    full_val_dataset = PhotoboxBBDataset("../train_data", is_train=False)

    for fold, (train_ids, val_ids) in enumerate(kfold.split(full_train_dataset)):
        print(f"Training fold {fold}\n" + "-" * 10 + "\n")

        train_dataset = torch.utils.data.Subset(full_train_dataset, train_ids)
        val_dataset = torch.utils.data.Subset(full_val_dataset, val_ids)

        base_model = get_model_instance_segmentation(
            PhotoboxBBDataset.num_classes,
            pretrained=PRETRAINED if not AGAR_TRAINING else False,
        ).to(device)
        if AGAR_TRAINING:
            base_model.load_state_dict(torch.load(agar_model_path))

        train_loader = prepare_dataloader(train_dataset, BATCH_SIZE, is_train=True)
        val_loader = prepare_dataloader(val_dataset, BATCH_SIZE, is_train=False)

        params = [p for p in base_model.parameters() if p.requires_grad]
        optimizer = AdamW(params, lr=0.005, weight_decay=0.01)

        writer = SummaryWriter(f"runs/{start_time_str}/fold_{fold}")

        trainer = Trainer(
            base_model,
            train_loader,
            val_loader,
            optimizer,
            device,
            writer,
            f"artifacts/{start_time_str}/fold_{fold}.pth",
        )

        trainer.train(EPOCHS_OWN)

    print(f"Final training run\n" + "-" * 10 + "\n")

    model = get_model_instance_segmentation(
        PhotoboxBBDataset.num_classes,
        pretrained=PRETRAINED if not AGAR_TRAINING else False,
    ).to(device)
    if AGAR_TRAINING:
        model.load_state_dict(torch.load(agar_model_path))

    train_loader = prepare_dataloader(full_train_dataset, BATCH_SIZE, is_train=True)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(params, lr=0.005, weight_decay=0.01)

    writer = SummaryWriter(f"runs/{start_time_str}/final")

    trainer = Trainer(
        model,
        train_loader,
        None,
        optimizer,
        device,
        writer,
        f"artifacts/{start_time_str}/final.pth",
    )

    trainer.train(EPOCHS_OWN)

In [ ]:
if __name__ == "__main__":
    device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    main("cuda:1")